# 2 — Scanpy PCA representations for the 10 samples

## Questions

1. Does conventional Scanpy PCA separate broad cell types in all 29,614 cells?
2. After subsetting, does CD4-only PCA resolve CD4 states?
3. Does CD8-only PCA resolve CD8 states?

Published annotations are used only to color and evaluate the unsupervised PCA
spaces; they are not inputs to normalization, HVG selection, PCA, or Leiden.

In [ ]:
from pathlib import Path
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"
FIGURES = RESULTS / "figures/demo"
TABLES = RESULTS / "tables"
EMBEDDINGS = RESULTS / "embeddings"
for directory in (FIGURES, TABLES, EMBEDDINGS):
    directory.mkdir(parents=True, exist_ok=True)
RAW_PATH = DATA / "processed/GSE205335_phase1_raw_counts.h5ad"
RANDOM_STATE = 0

source = sc.read_h5ad(RAW_PATH)
mask = source.obs["core.patient"].astype(str).eq("Core") & source.obs["Tissue origin"].astype(str).eq("Metastatic LN")
demo = source[mask].copy()
del source
assert demo.n_obs == 29_614 and demo.obs["Sample"].nunique() == 10
print(demo)

## 1. Shared Scanpy pipeline

In [ ]:
def build_pca(raw, label, resolution=0.6):
    work = raw.copy()
    sc.pp.highly_variable_genes(work, n_top_genes=2_000, flavor="seurat_v3", batch_key="Platform")
    sc.pp.normalize_total(work, target_sum=1e4)
    sc.pp.log1p(work)
    sc.pp.pca(work, n_comps=50, mask_var="highly_variable", random_state=RANDOM_STATE)
    sc.pp.neighbors(work, use_rep="X_pca", n_neighbors=15, n_pcs=50, random_state=RANDOM_STATE)
    sc.tl.umap(work, random_state=RANDOM_STATE)
    sc.tl.leiden(work, resolution=resolution, key_added=f"leiden_{label}",
                 random_state=RANDOM_STATE, flavor="igraph", n_iterations=2, directed=False)
    return work

all_cells = build_pca(demo, "all")
cd4 = build_pca(demo[demo.obs["lineage.sub"].astype(str).eq("CD4+ T cells")], "cd4", 0.8)
cd8 = build_pca(demo[demo.obs["lineage.sub"].astype(str).eq("CD8+ T cells")], "cd8", 0.8)
print("All/CD4/CD8:", all_cells.n_obs, cd4.n_obs, cd8.n_obs)

## 2. Three PCA/UMAP views

In [ ]:
plots = [
    (all_cells, "lineage.total", "All cells — broad lineages", "scanpy_all_cells_pca_umap.png"),
    (cd4, "celltype", "CD4-only — fine states", "scanpy_cd4_pca_umap.png"),
    (cd8, "celltype", "CD8-only — fine states", "scanpy_cd8_pca_umap.png"),
]
for obj, color, title, filename in plots:
    with plt.rc_context({"figure.figsize": (9, 6)}):
        sc.pl.umap(obj, color=color, title=title, show=False)
        plt.savefig(FIGURES / filename, dpi=220, bbox_inches="tight")
        plt.show()

## 3. Save compact PCA representations

In [ ]:
def save_representation(obj, path, leiden_key):
    out = ad.AnnData(X=obj.obsm["X_pca"].astype(np.float32), obs=obj.obs.copy())
    out.obsm["X_umap"] = obj.obsm["X_umap"].astype(np.float32)
    out.uns["representation"] = "Scanpy log-normalization, 2,000 batch-aware HVGs, 50-PC PCA"
    out.uns["leiden_key"] = leiden_key
    out.write_h5ad(path, compression="gzip")

save_representation(all_cells, EMBEDDINGS / "GSE205335_10sample_scanpy_pca_v1.h5ad", "leiden_all")
save_representation(cd4, EMBEDDINGS / "GSE205335_10sample_cd4_scanpy_pca_v1.h5ad", "leiden_cd4")
save_representation(cd8, EMBEDDINGS / "GSE205335_10sample_cd8_scanpy_pca_v1.h5ad", "leiden_cd8")
print("Saved three PCA representations")